In [1]:
import json 
import os
import requests #type: ignore
from datetime import datetime
import os
import json
import requests
from dotenv import load_dotenv
from collections import defaultdict
from typing import List, Dict
import datetime
from dateutil.parser import parse
from typing import List, Dict, Any, Optional, Set, DefaultDict
import re
import sys
from SRC import *

In [2]:
def load_model_metadata():
    """
    Loads and parses the model metadata from the JSON file.

    Returns:
        dict: The parsed JSON data from modelMetaData.json, or None if an error occurs.
    """
    try:        
        current_dir = os.getcwd() 
        project_root = os.path.dirname(current_dir) 

        if os.path.basename(os.getcwd()) == "Extract":
             project_root_path = os.path.dirname(os.getcwd())
        else: 
             project_root_path = os.getcwd()

        metadata_file_path = os.path.join(project_root_path, "SRC", "metaData", "modelMetaData.json")
        
        print(f"Attempting to load metadata from: {metadata_file_path}") 

        with open(metadata_file_path, 'r') as f:
            metadata = json.load(f)
        return metadata
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    

def setup_project_paths():
    """
    Adds the project's SRC directory to sys.path.
    This allows importing modules from packages within SRC, such as 'helpFolder'.
    """
    current_notebook_dir = os.getcwd() 
    project_root = os.path.dirname(current_notebook_dir)

    src_directory = os.path.join(project_root, "SRC")
    
    if src_directory not in sys.path:
        sys.path.insert(0, src_directory)
        print(f"Successfully added '{src_directory}' to sys.path.")
    else:
        print(f"'{src_directory}' is already in sys.path.")

setup_project_paths()
meta_data = load_model_metadata()
from helpFolder import *

Successfully added 'c:\Users\PREDATOR\Documents\PFEmaster\PFEVersion2\SRC' to sys.path.
Attempting to load metadata from: c:\Users\PREDATOR\Documents\PFEmaster\PFEVersion2\SRC\metaData\modelMetaData.json


In [3]:
all_rooms = get_all_rooms(meta_data['PMSInformation']['rooms_url'])
room_types, room_type_counts, total_rooms = get_room_types_from_rooms(all_rooms)

Token obtained successfully!
Rooms retrieved successfully!


In [4]:
room_types, room_type_counts, total_rooms

(['Imaginary Room', 'Family Room', 'Chambre Standard', 'Standard Room'],
 {'Imaginary Room': 67,
  'Family Room': 18,
  'Chambre Standard': 387,
  'Standard Room': 27},
 432)

In [5]:
detailed_metrics, overall_totals = run_analysis_in_intervals(
    start_date_str = meta_data['PMSInformation']['start_date_range'],
    end_date_str = meta_data['PMSInformation']['end_date_range'],
    initial_room_types= room_types,
    initial_room_type_counts= room_type_counts,
    api_templates= meta_data['PMSInformation']['api_templates']
)


=== Processing interval 1: 2024-05-01 to 2024-05-07 ===
Step 1: API templates defined.

Step 2: Sending requests to 3 APIs for date range 2024-05-01 to 2024-05-07.
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-05-01&to=2024-05-07&group=1&resastatus=2
Successfully fetched 288 original reservations.

Step 2.1: Filtering reservations for customer: 'COMPLI-Gratuit' and adjusting inventory.
  0 reservations removed for this customer.
  288 reservations remaining for analysis.

Step 3: Processing for 7 dates from 2024-05-01 to 2024-05-07.

Step 4: Grouping reservations by room type per date...
Reservations grouped by room type per date.

Step 5: Merging room type reservation lists within daily data using map: {'Chambre Standard': 'Standard Room'}
Room type reservation lists merged within daily data.

Step 6: Generating new list of room types and their inventory counts after merging names...
New room types (post-merge): ['Imaginary Room', 'Family

In [6]:
append_to_json_file(detailed_metrics, meta_data['PMSInformation']['final_data_path'])

Data saved to ../Data/TransformedPMSData/final_reservation_metrics.json
